# Brain-state transitions vs. phenotype — the volcano

Reads the tables written by `fmri-decomp transitions` and asks whether any
single transition cell tracks a phenotype column.

1. the per-subject transition table, as a matrix you can look at
2. a volcano: one dot per transition cell, across every state set

**Model selection lives in a script, not here.** It is state sets x models x
fold seeds — hundreds of fits, minutes to tens of minutes, and it blocks the
kernel while it runs:

```bash
sbatch slurm/05_select.sbatch additional_HADS_anx_category -- --models ridge hgb
```

which writes `outputs/bstm_selection/target=<t>/` — scores, a ranking, the
comparison figures and the refit model artifacts.

**What this notebook can and cannot tell you.** The volcano asks the *weakest*
version of the question: does one cell, on its own, track the phenotype. Each
cell is a noisy fraction of ~74 transitions, so if transition structure carries
signal it is probably spread across many cells rather than concentrated in one.
A null volcano beside a model that beats baseline is coherent, and says the
signal is distributed.

Nothing here writes to `outputs/`.


In [ ]:
# ---- parameters -------------------------------------------------------------
import os
from pathlib import Path

ROOT = Path(os.environ.get("FMRIDECOMP_OUTPUTS",
                           "/project/6008063/tamires/DecomposingfMRI/outputs"))

# Phenotype: one row per subject. `PHENO_ID` is joined to the `sub=` partition
# key, so it must hold the bare id as a STRING -- CC110019, never parsed as a
# number.
# Cam-CAN splits what we need across two release tables, both keyed by CCID and
# both 2,681 rows: the HADS categories are in approved_data.tsv, Age and Sex in
# standard_data.csv. They are read and merged on CCID, so a missing covariate is
# a missing FILE rather than a missing column.
CAMCAN = Path("/project/6008063/tamires/cohorts/camcan/dataman/useraccess/"
              "opendata/paule_toussaint_camcan01870")
PHENO_FILES = [(CAMCAN / "approved_data.tsv", "\t"),     # the HADS targets
               (CAMCAN / "standard_data.csv", ",")]      # Age, Sex, education
PHENO_ID = "CCID"
PHENO_COHORT = "camcan"

TARGETS = ["additional_HADS_anx_category", "additional_HADS_dep_category"]
TARGET = TARGETS[0]       # switch to TARGETS[1] and re-run the notebook

# Ordered, low to high. Position in this list IS the code: Normal=0 ... Severe=3.
# A value not in the list becomes NaN and is reported -- a fifth category, or a
# stray spelling, must be visible rather than silently dropped.
ORDINAL_LEVELS = ["Normal", "Mild", "Moderate", "Severe"]

# The estimator is NOT set here -- model selection is `fmri-decomp select`,
# which takes --models ridge hgb lgbm logistic. This notebook only correlates.

# Nuisance. age is not optional: camcan spans 18-88, and both brain dynamics and
# HADS move with it, so without it the volcano reports age effects wearing a
# HADS label. n_transitions varies because censoring removed windows unevenly.
# Capitalised, because that is what standard_data.csv calls them. Resolved
# case-insensitively below anyway, and what was matched is printed.
# `Yrs Education` is also in that file if you want it here later.
COVARIATES = ["Age", "Sex", "n_transitions"]
CATEGORICAL = ["Sex"]            # one-hot; everything else is forced numeric

P_COLUMN = "p_joint"      # or "p_cond" -- see section 1
N_PERM = 1000
SEEDS = [0, 1, 2, 3, 4]   # the state-set ranking is repeated over these, to see
                          # whether the winner is stable or noise

In [ ]:
import itertools, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

TRANS = ROOT / "transitions"
if not TRANS.is_dir():
    raise SystemExit(f"{TRANS} does not exist -- run `fmri-decomp transitions` first")

# Every state set that actually has tables, discovered from the tree.
sets = pd.DataFrame(
    [{"atlas": p.parts[-5].split("=")[1], "window_s": p.parts[-4].split("=")[1],
      "states": p.parts[-3].split("=")[1], "cohort": p.parts[-2].split("=")[1],
      "path": p.parent}
     for p in TRANS.rglob("subjects.parquet")])
sets["K"] = sets["states"].str.rsplit("_", n=1).str[1].astype(int)
sets["window_s_num"] = sets["window_s"].astype(float)

print(f"{len(sets)} (state set x cohort) table(s) on disk\n")
print(sets.groupby(["atlas", "window_s_num", "K"]).size()
          .rename("cohorts").reset_index().to_string(index=False))

## 1. The transition table

One row per subject, one table. Columns in three blocks:

```
cohort task sub | 0->0 0->1 ... (every K*K cell) | switch_rate ... | provenance
```

Cell values are `p_joint` — that subject's count for `i->j` divided by all of
their transitions, so the cells sum to 1 across the row. Unobserved transitions
are exact zeros, not missing.

Nothing is lost by storing one normalisation: `count = p_joint * n_transitions`
exactly, and `p_cond` (the Markov probability, each row summing to 1) is the
row-wise renormalisation below.

In [ ]:
def load(atlas, window_s, states, cohort):
    """The one table for a (state set, cohort). One row per subject."""
    return pd.read_parquet(TRANS / f"atlas={atlas}" / f"window_s={window_s}"
                           / f"states={states}" / f"cohort={cohort}"
                           / "subjects.parquet")

def cell_cols(d):
    """The transition columns, in (from, to) order."""
    return sorted((c for c in d.columns if "->" in c),
                  key=lambda c: tuple(int(x) for x in c.split("->")))

def as_p_cond(d):
    """Cells renormalised per from-state: the Markov transition probability."""
    cols, K = cell_cols(d), int(d["n_states"].iloc[0])
    M = d[cols].to_numpy(float).reshape(len(d), K, K)
    M = M * d["n_transitions"].to_numpy(float)[:, None, None]      # back to counts
    rowsum = M.sum(axis=2, keepdims=True)
    out = np.divide(M, rowsum, out=np.zeros_like(M), where=rowsum > 0)
    return pd.DataFrame(out.reshape(len(d), K * K), columns=cols, index=d.index)

demo = sets[sets["cohort"] == PHENO_COHORT].sort_values(["K", "window_s_num"]).iloc[0]
print(f"showing: atlas={demo.atlas}  window_s={demo.window_s}  "
      f"{demo.states}  cohort={demo.cohort}")
d0 = load(demo.atlas, demo.window_s, demo.states, demo.cohort)
cols0 = cell_cols(d0)
print(f"{d0.shape[0]} subject(s) x {d0.shape[1]} column(s)  ({len(cols0)} cells)")
print(f"zeros among the cells: {100 * (d0[cols0] == 0).to_numpy().mean():.1f}%")
d0[["sub"] + cols0[:8] + ["n_transitions", "switch_rate", "mean_dwell_s"]].head(8).round(3)

## 2. Volcano — every transition cell, every state set

One dot per `(state set, transition cell)`. The test runs **across subjects**:
each subject contributes one value of `p` for that cell, and the question is
whether it tracks the phenotype. So **n = subjects, and subjects are
independent** — the overlapping windows live inside each subject's number and
cost precision, not validity.

Both axes come from a **partial** correlation: phenotype and cell are each
residualised on the covariates first, so what is plotted is the association
that age, sex and transition count do not already explain.

Two thresholds are drawn. **BH-FDR** at 5% on the analytic p-values, and a
**permutation FWER** line from the null distribution of the largest |r| over
every cell — the second is the stricter one and needs no independence
assumption across cells.

Cells that are zero for every subject have no variance, cannot be tested, and
are dropped before either correction, so they do not inflate the family.


In [ ]:
def design(cov: pd.DataFrame) -> np.ndarray:
    """Covariates -> a numeric design matrix with an intercept."""
    X = pd.get_dummies(cov, drop_first=True, dummy_na=False).astype(float)
    X = X.fillna(X.mean())
    return np.column_stack([np.ones(len(X)), X.to_numpy()])

def residualise(Y: np.ndarray, X: np.ndarray) -> np.ndarray:
    """Columns of Y with the span of X projected out."""
    beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
    return Y - X @ beta

def _z(A):
    """Standardise columns, leaving zero-variance ones as exact zeros.

    `out=` is not optional: np.divide with `where=` and no `out` leaves the
    masked entries UNINITIALISED, so a flat column would carry whatever was in
    memory. They are dropped downstream by `keep`, but a correct value beats
    relying on that.
    """
    A = A - A.mean(0)
    sd = A.std(0)
    return np.divide(A, sd, out=np.zeros_like(A), where=sd > 0), sd

def partial_corr_scan(cells_wide: pd.DataFrame, y: np.ndarray, X: np.ndarray,
                      n_perm: int = N_PERM, seed: int = 0):
    """Partial correlation of every column against y, plus a permutation null.

    Vectorised on purpose: residualise once, standardise once, then every
    correlation is one matrix product. The permutation null shuffles the
    residualised phenotype, which is the exchangeable quantity once the
    covariates are out.
    """
    A = residualise(cells_wide.to_numpy(float), X)
    yr = residualise(y.reshape(-1, 1), X).ravel()
    Az, sd = _z(A)
    keep = sd > 0                                  # no variance -> untestable
    Az = Az[:, keep]
    yz = (yr - yr.mean()) / yr.std()
    n = len(yz)
    r = (Az.T @ yz) / n

    df = n - X.shape[1] - 1
    with np.errstate(divide="ignore", invalid="ignore"):
        t = r * np.sqrt(df / (1 - r ** 2))
    p = 2 * stats.t.sf(np.abs(t), df)

    rng = np.random.default_rng(seed)
    max_null = np.empty(n_perm)
    for b in range(n_perm):
        max_null[b] = np.abs((Az.T @ rng.permutation(yz)) / n).max()

    out = pd.DataFrame({"transition": cells_wide.columns[keep], "r": r, "p": p})
    out["p_fdr"] = stats.false_discovery_control(out["p"], method="bh")
    return out, float(np.quantile(max_null, 0.95)), int((~keep).sum())

In [ ]:
# ---- phenotype: both release tables, merged on CCID -------------------------
frames = []
for path, sep in PHENO_FILES:
    if not path.exists():
        raise SystemExit(f"{path} does not exist -- check PHENO_FILES.")
    f = pd.read_csv(path, sep=sep, dtype={PHENO_ID: str})
    if PHENO_ID not in f.columns:
        raise SystemExit(f"{PHENO_ID!r} is not in {path.name}: "
                         f"{list(f.columns)[:20]}")
    f[PHENO_ID] = f[PHENO_ID].astype(str).str.strip().str.upper()
    print(f"  {path.name:<22} {len(f):>6,} row(s)  {len(f.columns):>4} column(s)")
    frames.append(f)

pheno = frames[0]
for f in frames[1:]:
    # Columns present in both would otherwise become _x/_y silently. Keep the
    # first file's copy and say which were dropped.
    dup = [c for c in f.columns if c in pheno.columns and c != PHENO_ID]
    if dup:
        print(f"  (also in an earlier file, keeping the first: {dup[:8]})")
    pheno = pheno.merge(f.drop(columns=dup), on=PHENO_ID, how="outer")
pheno = pheno.rename(columns={PHENO_ID: "sub"})
print(f"  merged -> {len(pheno):,} row(s), {len(pheno.columns):,} column(s)")

# Resolve the requested names case-insensitively, and say what was matched --
# `Age` vs `age` cost a run once.
lower = {c.strip().lower(): c for c in pheno.columns}
resolved, missing = {}, []
for want in [TARGET] + [c for c in COVARIATES if c != "n_transitions"]:
    hit = lower.get(want.strip().lower())
    (resolved.__setitem__(want, hit) if hit else missing.append(want))
if missing:
    near = [c for c in pheno.columns
            if any(k in c.lower() for k in ("hads", "age", "sex", "gender"))]
    raise SystemExit(f"not in any of PHENO_FILES: {missing}\nclose names: {near[:14]}")
for want, got in resolved.items():
    if got != want:
        pheno[want] = pheno[got]
        print(f"  matched {want!r} -> column {got!r}")

# Ordered categories -> integer codes. Not pd.factorize, which numbers by order
# of appearance; the clinical ordering has to be the one we assert.
raw = pheno[TARGET].astype("string").str.strip()
codes = raw.map({lvl: k for k, lvl in enumerate(ORDINAL_LEVELS)})
unknown = sorted(set(raw.dropna()) - set(ORDINAL_LEVELS))
if unknown:
    print(f"  WARNING: value(s) not in ORDINAL_LEVELS, now NaN: {unknown}")
pheno["y"] = codes.astype("Float64")

print(f"\n  {TARGET}")
dist = raw.value_counts(dropna=False).reindex(ORDINAL_LEVELS + [np.nan])
for lvl, n in dist.items():
    code = ORDINAL_LEVELS.index(lvl) if lvl in ORDINAL_LEVELS else "-"
    n = 0 if pd.isna(n) else int(n)
    print(f"    {str(lvl):<10} code {str(code):<3} {n:>6,}")
pheno = pheno.dropna(subset=["y"])
pheno["y"] = pheno["y"].astype(int)
print(f"  -> {len(pheno):,} row(s) with a usable label")

# Everything numeric except the declared categoricals. Cam-CAN writes the
# string "NaN" in some columns, which to_numeric turns into a real NaN.
for c in COVARIATES:
    if c in pheno.columns and c not in CATEGORICAL:
        pheno[c] = pd.to_numeric(pheno[c], errors="coerce")

cov_cols = [c for c in COVARIATES if c != "n_transitions"]
before = len(pheno)
pheno = pheno.dropna(subset=cov_cols)
if len(pheno) < before:
    print(f"  dropped {before - len(pheno):,} row(s) with a missing covariate "
          f"({cov_cols}) -> {len(pheno):,} left")
print("\n  covariates as read:")
for c in cov_cols:
    s = pheno[c]
    print(f"    {c:<16} " + (f"{s.value_counts().to_dict()}" if c in CATEGORICAL
                             else f"min {s.min():.1f}  median {s.median():.1f}  "
                                  f"max {s.max():.1f}"))


def feature_frame(atlas, window_s, states, cohort=PHENO_COHORT):
    """One transition table joined to phenotype on `sub`. Returns (frame, cells)."""
    t = load(atlas, window_s, states, cohort)
    if P_COLUMN == "p_cond":
        t[cell_cols(t)] = as_p_cond(t)
    t["sub"] = t["sub"].astype(str).str.strip().str.upper()
    d = t.merge(pheno, on="sub", how="inner", suffixes=("", "_pheno"))
    if d.empty:
        raise SystemExit(
            "the phenotype join matched NOTHING -- the identifiers differ.\n"
            f"  transitions sub e.g. {t['sub'].iloc[:3].tolist()}\n"
            f"  phenotype   sub e.g. {pheno['sub'].iloc[:3].tolist()}")
    return d, cell_cols(t)


grid = (sets[sets["cohort"] == PHENO_COHORT]
        .sort_values(["atlas", "K", "window_s_num"]))
print(f"\nscanning {len(grid)} state set(s) against {TARGET}\n")

scans, fwer = [], {}
for t in grid.itertuples():
    d, cols = feature_frame(t.atlas, t.window_s, t.states)
    if len(d) < 30:
        print(f"  skip {t.atlas}/{t.window_s}s/K={t.K}: only {len(d)} subject(s)")
        continue
    X = design(d[COVARIATES])
    # The target is a 4-level ordinal, so rank it: this makes the scan a PARTIAL
    # SPEARMAN, which does not assume the gap Normal->Mild equals Moderate->Severe.
    y = stats.rankdata(d["y"].to_numpy(float))
    res, thr, n_flat = partial_corr_scan(d[cols], y, X)
    scans.append(res.assign(atlas=t.atlas, window_s=t.window_s_num, K=t.K,
                            n=len(d)))
    fwer[(t.atlas, t.window_s_num, t.K)] = thr
    print(f"  {t.atlas:>14} {t.window_s:>4}s K={t.K:<3} n={len(d):<4} "
          f"{len(res):>5} testable cell(s) ({n_flat} flat)  "
          f"perm |r| 95% = {thr:.3f}")

volc = pd.concat(scans, ignore_index=True)
volc["p_fdr_all"] = stats.false_discovery_control(volc["p"], method="bh")
print(f"\nfamily: {len(volc):,} testable cell(s) across every state set")
print(f"surviving BH-FDR 5% over the whole family: {(volc['p_fdr_all'] < .05).sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
colour = {8: "tab:blue", 27: "tab:orange"}
for K, g in volc.groupby("K"):
    ax.scatter(g["r"], -np.log10(g["p"]), s=6, alpha=.35,
               c=colour.get(K, "grey"), label=f"K={K}  ({len(g):,} cells)")

sig = volc[volc["p_fdr_all"] < .05]
if len(sig):
    ax.scatter(sig["r"], -np.log10(sig["p"]), s=26, facecolors="none",
               edgecolors="k", linewidths=.8, label=f"BH-FDR 5% ({len(sig)})")
    for t in sig.nsmallest(8, "p").itertuples():
        ax.annotate(f"{t.transition}\n{t.atlas} {t.window_s:g}s K={t.K}",
                    (t.r, -np.log10(t.p)), fontsize=6,
                    xytext=(4, 2), textcoords="offset points")

if len(volc):
    worst = volc.loc[volc["p_fdr_all"] < .05, "p"].max() if len(sig) else None
    if worst:
        ax.axhline(-np.log10(worst), ls="--", lw=.8, c="k",
                   label="BH-FDR 5% threshold")
ax.axvline(0, lw=.6, c="grey")
ax.set_xlabel(f"partial r  (adjusted for {', '.join(COVARIATES)})")
ax.set_ylabel(r"$-\log_{10}$ p")
ax.set_title(f"{TARGET} vs transition cells — every state set\n"
             f"family = {len(volc):,} tests, n = subjects (independent)", fontsize=10)
ax.legend(fontsize=7, loc="upper left")
fig.tight_layout()

print(volc.nsmallest(12, "p")[["atlas","window_s","K","transition","r","p","p_fdr_all"]]
          .to_string(index=False))

## Reading the volcano

* **The V shape is not a result.** With n fixed across every test, p is a
  deterministic function of |r|, so every point must lie on one smooth curve.
  A signal shows as points pushed far out along the arms, never off the curve.
* **Two corrections, two questions.** `p_fdr_all` controls the false discovery
  rate over the whole family printed in the title. `fwer[(atlas, window, K)]` is
  a permutation threshold on the largest |r| *within one state set*. A cell can
  clear the second and fail the first; that makes it a candidate, not a finding.
* **Cells are compositional, so hits come in sprays.** Each subject's row sums
  to 1, so inflating one cell deflates the others. One planted effect produced
  3 surviving cells when weak and all 64 when strong. "How many survived" is not
  "how many findings" — read which cell is strongest and treat its neighbours as
  the same result seen sideways.
* **Flat cells are dropped before correction.** A cell that is zero for every
  subject has no variance and cannot be tested, so it does not inflate the
  family. The printed count says how many went.
* **Age is adjusted out, and that is conservative.** camcan spans 18-88 and both
  dynamics and HADS move with age, so any anxiety signal that happens to be
  age-shaped is removed. That is the right call for a clean claim.

To check the strongest cells against their own state set's permutation
threshold:

```python
top = volc.nsmallest(10, "p").copy()
top["perm_thr"] = [fwer[(r.atlas, r.window_s, r.K)] for r in top.itertuples()]
top["beats_perm"] = top["r"].abs() > top["perm_thr"]
print(top[["atlas", "window_s", "K", "transition", "r", "p",
           "p_fdr_all", "perm_thr", "beats_perm"]])
```
